<a href="https://colab.research.google.com/github/imenk-cloud/my-novel/blob/main/vision_language_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
%cd /content
!git clone https://github.com/RLS-ResearchLab/RLS-Entrance-Challenge.git

/content
Cloning into 'RLS-Entrance-Challenge'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 59 (delta 11), reused 53 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 56.33 KiB | 6.26 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [11]:
%cd /content/RLS-Entrance-Challenge
!pwd
!ls

/content/RLS-Entrance-Challenge
/content/RLS-Entrance-Challenge
AI_USAGE.md  benchmarks  experiments	   LICENSE    report		src
assets	     configs	 generate_data.py  README.md  requirements.txt	tests


In [12]:
!python generate_data.py --seed 42 --out-dir data --train-n 20000 --val-n 2000 --test-n 2000 --heldout-n 2000

train: 100% 20000/20000 [00:12<00:00, 1544.79it/s]
train: 20000 samples in 13.7s -> data/train.pt
val: 100% 2000/2000 [00:01<00:00, 1045.19it/s]
val: 2000 samples in 2.2s -> data/val.pt
test: 100% 2000/2000 [00:02<00:00, 931.32it/s]
test: 2000 samples in 2.9s -> data/test.pt
test_heldout: 100% 2000/2000 [00:02<00:00, 721.83it/s] 
test_heldout: 2000 samples in 2.9s -> data/test_heldout.pt
meta -> data/meta.json


In [13]:
!find data -maxdepth 3 | head -40

data
data/train.pt
data/val.pt
data/meta.json
data/test_heldout.pt
data/test.pt


In [14]:
import torch, json

# Regarde les métadonnées d'abord
with open("data/meta.json") as f:
    meta = json.load(f)
print(meta)

{'seed': 42, 'image_size': 64, 'colors': ['red', 'green', 'blue', 'yellow'], 'shapes': ['circle', 'square', 'triangle', 'cross'], 'sizes': ['small', 'large'], 'relations': ['leftof', 'rightof', 'above', 'below'], 'held_out_pairs': [['green', 'triangle'], ['yellow', 'cross'], ['blue', 'square']], 'vocabulary': ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '<eos>'], 'splits': {'train': 20000, 'val': 2000, 'test': 2000, 'test_heldout': 2000}, 'generated_at': '2026-09-29T22:45:51Z'}


In [15]:
data = torch.load("data/train.pt")
print(type(data))

# Si c'est un dict :
if isinstance(data, dict):
    for k, v in data.items():
        print(k, type(v), getattr(v, "shape", None))

<class 'dict'>
images <class 'torch.Tensor'> torch.Size([20000, 3, 64, 64])
words <class 'list'> None


In [16]:
print(data["images"].dtype)
print(data["images"].min(), data["images"].max())
print(data["words"][:5])
print(meta)  # affiche tout le meta.json, pas juste le début tronqué

torch.uint8
tensor(0, dtype=torch.uint8) tensor(255, dtype=torch.uint8)
['largegreensquareabovesmallredtriangle', 'smallyellowcirclebelowsmallgreencross', 'largeyellowtrianglebelowsmallyellowcircle', 'largebluetriangleleftofsmallbluecross', 'smallredtriangle']
{'seed': 42, 'image_size': 64, 'colors': ['red', 'green', 'blue', 'yellow'], 'shapes': ['circle', 'square', 'triangle', 'cross'], 'sizes': ['small', 'large'], 'relations': ['leftof', 'rightof', 'above', 'below'], 'held_out_pairs': [['green', 'triangle'], ['yellow', 'cross'], ['blue', 'square']], 'vocabulary': ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '<eos>'], 'splits': {'train': 20000, 'val': 2000, 'test': 2000, 'test_heldout': 2000}, 'generated_at': '2026-09-29T22:45:51Z'}


In [17]:
import pprint
pprint.pprint(meta)

{'colors': ['red', 'green', 'blue', 'yellow'],
 'generated_at': '2026-09-29T22:45:51Z',
 'held_out_pairs': [['green', 'triangle'],
                    ['yellow', 'cross'],
                    ['blue', 'square']],
 'image_size': 64,
 'relations': ['leftof', 'rightof', 'above', 'below'],
 'seed': 42,
 'shapes': ['circle', 'square', 'triangle', 'cross'],
 'sizes': ['small', 'large'],
 'splits': {'test': 2000, 'test_heldout': 2000, 'train': 20000, 'val': 2000},
 'vocabulary': ['a',
                'b',
                'c',
                'd',
                'e',
                'f',
                'g',
                'h',
                'i',
                'j',
                'k',
                'l',
                'm',
                'n',
                'o',
                'p',
                'q',
                'r',
                's',
                't',
                'u',
                'v',
                'w',
                'x',
                'y',
             

In [18]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# ---------- Tokenizer ----------
class CharTokenizer:
    def __init__(self):
        self.chars = list("abcdefghijklmnopqrstuvwxyz")
        self.stoi = {c: i for i, c in enumerate(self.chars)}
        self.stoi["<eos>"] = 26
        self.itos = {i: c for c, i in self.stoi.items()}
        self.eos_id = self.stoi["<eos>"]
        self.vocab_size = 27  # 26 lettres + eos

    def encode(self, word):
        # renvoie une liste d'ids, avec <eos> à la fin
        return [self.stoi[c] for c in word] + [self.eos_id]

    def decode(self, ids):
        out = []
        for i in ids:
            if i == self.eos_id:
                break
            out.append(self.itos.get(i, "?"))
        return "".join(out)

tok = CharTokenizer()
print(tok.encode("cat"))
print(tok.decode(tok.encode("cat")))

# ---------- Dataset ----------
class ShapeScenesDataset(Dataset):
    def __init__(self, pt_path, tokenizer):
        data = torch.load(pt_path)
        self.images = data["images"]      # (N, 3, 64, 64) uint8
        self.words = data["words"]        # list[str]
        self.tok = tokenizer

    def __len__(self):
        return len(self.words)

    def __getitem__(self, idx):
        img = self.images[idx].float() / 255.0   # normalise 0-1, (3,64,64)
        ids = torch.tensor(self.tok.encode(self.words[idx]), dtype=torch.long)
        return img, ids

# ---------- Collate (gère les longueurs de mots différentes) ----------
PAD_ID = -100  # valeur utilisée par ignore_index dans CrossEntropyLoss

def collate_fn(batch):
    imgs, ids_list = zip(*batch)
    imgs = torch.stack(imgs)  # (B, 3, 64, 64)
    lengths = [len(ids) for ids in ids_list]
    max_len = max(lengths)
    padded = torch.full((len(ids_list), max_len), PAD_ID, dtype=torch.long)
    for i, ids in enumerate(ids_list):
        padded[i, :len(ids)] = ids
    return imgs, padded, torch.tensor(lengths)

# ---------- Test rapide ----------
train_ds = ShapeScenesDataset("data/train.pt", tok)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)

imgs, ids, lengths = next(iter(train_loader))
print("images:", imgs.shape, imgs.dtype)
print("ids:", ids.shape)
print("lengths:", lengths)
print("un exemple décodé:", tok.decode(ids[0].tolist()))

[2, 0, 19, 26]
cat
images: torch.Size([8, 3, 64, 64]) torch.float32
ids: torch.Size([8, 42])
lengths: tensor([15, 37, 42, 40, 42, 38, 17, 37])
un exemple décodé: smallbluecross


In [19]:
import torch.nn as nn

class CNNEncoder(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.adapter = nn.Linear(128, d_model)

    def forward(self, x):
        feat = self.conv(x)
        B, C, H, W = feat.shape
        feat = feat.flatten(2)
        feat = feat.transpose(1, 2)
        visual_tokens = self.adapter(feat)
        return visual_tokens

encoder = CNNEncoder(d_model=128)
out = encoder(imgs)
print("shape des tokens visuels:", out.shape)
print("dtype:", out.dtype)

shape des tokens visuels: torch.Size([8, 64, 128])
dtype: torch.float32


In [20]:
import torch
import torch.nn as nn
import math

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model doit être divisible par n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads  # dimension par tête

        # Une seule projection linéaire pour Q, K, V (plus efficace que 3 séparées)
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        # x: (B, T, d_model)  -- T = longueur de la séquence (visual tokens + lettres)
        B, T, _ = x.shape

        qkv = self.qkv_proj(x)  # (B, T, 3*d_model)
        q, k, v = qkv.chunk(3, dim=-1)  # chacun (B, T, d_model)

        # Split en plusieurs têtes: (B, T, d_model) -> (B, n_heads, T, d_head)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        # Scaled dot-product attention
        # scores: (B, n_heads, T, T)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)

        if mask is not None:
            # mask: (T, T) ou (B, 1, T, T), True/1 = positions autorisées, ou -inf directement
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attn = torch.softmax(scores, dim=-1)  # (B, n_heads, T, T)
        out = attn @ v  # (B, n_heads, T, d_head)

        # Merge heads: (B, n_heads, T, d_head) -> (B, T, d_model)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        out = self.out_proj(out)
        return out

# ---------- Test rapide ----------
d_model = 128
n_heads = 4
T = 64 + 42  # 64 tokens visuels + jusqu'à 42 lettres (exemple)

attn_layer = MultiHeadSelfAttention(d_model, n_heads)
x = torch.randn(2, T, d_model)  # batch factice
y = attn_layer(x)
print("input:", x.shape, "-> output:", y.shape)

input: torch.Size([2, 106, 128]) -> output: torch.Size([2, 106, 128])


In [21]:
def build_mask(num_visual, num_text, device):
    """
    Retourne un masque (T, T) où T = num_visual + num_text.
    1 = autorisé à regarder, 0 = interdit.
    Règles:
    - visual <-> visual : tout autorisé (bidirectionnel)
    - text -> visual : autorisé (le texte peut toujours voir l'image)
    - text -> text : causal (ne voit pas le futur)
    - visual -> text : interdit (l'image ne dépend pas des lettres générées)
    """
    T = num_visual + num_text
    mask = torch.zeros(T, T)

    # visual voit tout le bloc visual
    mask[:num_visual, :num_visual] = 1

    # text voit tout le bloc visual
    mask[num_visual:, :num_visual] = 1

    # text voit le texte de façon causale (triangulaire inférieure)
    causal = torch.tril(torch.ones(num_text, num_text))
    mask[num_visual:, num_visual:] = causal

    # visual ne voit jamais le texte -> reste à 0 (déjà initialisé)

    return mask.to(device)

# ---------- Test rapide ----------
m = build_mask(num_visual=4, num_text=3, device="cpu")
print(m)
print(m.shape)  # attendu (7,7)

tensor([[1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1.]])
torch.Size([7, 7])


In [22]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model * mlp_ratio),
            nn.GELU(),
            nn.Linear(d_model * mlp_ratio, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, mask=None):
        # Pre-LayerNorm + résiduel autour de l'attention
        x = x + self.attn(self.ln1(x), mask=mask)
        # Pre-LayerNorm + résiduel autour du MLP
        x = x + self.mlp(self.ln2(x))
        return x

# ---------- Test rapide ----------
block = DecoderBlock(d_model=128, n_heads=4)
x = torch.randn(2, 106, 128)
mask = build_mask(num_visual=64, num_text=42, device="cpu")
y = block(x, mask=mask)
print(y.shape)  # attendu (2, 106, 128)

torch.Size([2, 106, 128])


In [23]:
import torch.nn.functional as F

# ---------- Modèle complet ----------
class TinyVLM(nn.Module):
    def __init__(self, vocab_size=27, d_model=128, n_heads=4, n_layers=4, num_visual=64, max_text_len=45):
        super().__init__()
        self.encoder = CNNEncoder(d_model=d_model)
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(num_visual + max_text_len, d_model)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        self.num_visual = num_visual

    def forward(self, images, text_ids):
        # images: (B,3,64,64) | text_ids: (B, T_text) déjà décalés (input, pas target)
        B = images.shape[0]
        visual_tokens = self.encoder(images)                     # (B, 64, d_model)
        text_tokens = self.token_emb(text_ids.clamp(min=0))       # (B, T_text, d_model)
        x = torch.cat([visual_tokens, text_tokens], dim=1)        # (B, 64+T_text, d_model)

        T = x.shape[1]
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        x = x + self.pos_emb(positions)

        mask = build_mask(self.num_visual, T - self.num_visual, x.device)
        for block in self.blocks:
            x = block(x, mask=mask)
        x = self.ln_f(x)
        logits = self.head(x)                                     # (B, T, vocab_size)
        return logits[:, self.num_visual:, :]                     # on ne garde que les positions "texte"

# ---------- E0: overfit un seul batch (sanity check) ----------
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinyVLM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

imgs, ids, lengths = next(iter(train_loader))
imgs, ids = imgs.to(device), ids.to(device)

# input = tous les tokens sauf le dernier | target = tous sauf le premier (décalage classique)
input_ids = ids[:, :-1].clone()
input_ids[input_ids == PAD_ID] = 0  # remplace le padding par un id valide pour l'embedding
target_ids = ids[:, 1:]

losses = []
for step in range(300):
    logits = model(imgs, input_ids)
    loss = F.cross_entropy(logits.reshape(-1, 27), target_ids.reshape(-1), ignore_index=PAD_ID)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 50 == 0:
        print(f"step {step}, loss {loss.item():.4f}")

print("Loss finale:", losses[-1])

step 0, loss 3.3824
step 50, loss 0.3451
step 100, loss 0.0352
step 150, loss 0.0161
step 200, loss 0.0108
step 250, loss 0.0079
Loss finale: 0.006375941447913647


In [24]:
@torch.no_grad()
def generate(model, image, tokenizer, max_len=45, device="cpu"):
    model.eval()
    image = image.unsqueeze(0).to(device)  # (1, 3, 64, 64)
    generated = []
    input_ids = torch.zeros((1, 1), dtype=torch.long, device=device)  # id factice de départ

    for step in range(max_len):
        # on ne donne que les lettres déjà générées (pas de vraie première lettre imposée,
        # le premier pas prédit directement la 1ère lettre à partir des visual tokens)
        if step == 0:
            cur_input = torch.zeros((1, 0), dtype=torch.long, device=device)  # séquence vide
        else:
            cur_input = torch.tensor([generated], dtype=torch.long, device=device)

        # Le modèle attend au moins 1 token texte pour le masque -> on gère le cas step=0 à part
        if cur_input.shape[1] == 0:
            cur_input = torch.zeros((1, 1), dtype=torch.long, device=device)
            logits = model(image, cur_input)
            next_id = logits[0, 0].argmax().item()
        else:
            logits = model(image, cur_input)
            next_id = logits[0, -1].argmax().item()

        if next_id == tokenizer.eos_id:
            break
        generated.append(next_id)

    model.train()
    return tokenizer.decode(generated)

# ---------- Test sur une image du batch qu'on vient d'overfitter ----------
test_img = imgs[0].cpu()
predicted_word = generate(model, test_img, tok, device=device)
true_word = tok.decode(ids[0].tolist())

print("Mot prédit  :", predicted_word)
print("Mot attendu :", true_word)

Mot prédit  : rergeredtriangle
Mot attendu : largeredtriangle


In [25]:
import torch.nn.functional as F

def test_attention_equivalence():
    torch.manual_seed(0)
    B, T, d_model, n_heads = 2, 10, 32, 4
    d_head = d_model // n_heads

    x = torch.randn(B, T, d_model)
    my_attn = MultiHeadSelfAttention(d_model, n_heads)

    # on récupère Q,K,V de NOTRE implémentation pour les réutiliser dans la version PyTorch
    qkv = my_attn.qkv_proj(x)
    q, k, v = qkv.chunk(3, dim=-1)
    q = q.view(B, T, n_heads, d_head).transpose(1, 2)
    k = k.view(B, T, n_heads, d_head).transpose(1, 2)
    v = v.view(B, T, n_heads, d_head).transpose(1, 2)

    causal_mask = torch.tril(torch.ones(T, T)).bool()  # masque causal simple pour le test

    # Notre version (recalcul manuel avec le même masque)
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_head)
    scores = scores.masked_fill(~causal_mask, float("-inf"))
    attn = torch.softmax(scores, dim=-1)
    out_manual = (attn @ v).transpose(1, 2).contiguous().view(B, T, d_model)
    out_manual = my_attn.out_proj(out_manual)

    # Version PyTorch officielle
    out_pytorch = F.scaled_dot_product_attention(q, k, v, attn_mask=causal_mask)
    out_pytorch = out_pytorch.transpose(1, 2).contiguous().view(B, T, d_model)
    out_pytorch = my_attn.out_proj(out_pytorch)

    max_diff = (out_manual - out_pytorch).abs().max().item()
    print("Différence max:", max_diff)
    assert max_diff < 1e-4, "Attention non équivalente !"
    print("✅ Test d'équivalence attention: PASSED")

test_attention_equivalence()

Différence max: 1.4901161193847656e-07
✅ Test d'équivalence attention: PASSED


In [ ]:
import time

# Nouveau modèle frais pour un vrai entraînement
model = TinyVLM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

val_ds = ShapeScenesDataset("data/val.pt", tok)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, collate_fn=collate_fn)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, collate_fn=collate_fn)

def prepare_batch(imgs, ids):
    imgs = imgs.to(device)
    ids = ids.to(device)
    input_ids = ids[:, :-1].clone()
    input_ids[input_ids == PAD_ID] = 0
    target_ids = ids[:, 1:]
    return imgs, input_ids, target_ids

train_losses, val_losses = [], []
N_EPOCHS = 3  # vu le temps limité, on reste modeste mais suffisant pour montrer une vraie courbe

t0 = time.time()
for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss = 0
    for imgs_b, ids_b, _ in train_loader:
        imgs_b, input_ids, target_ids = prepare_batch(imgs_b, ids_b)
        logits = model(imgs_b, input_ids)
        loss = F.cross_entropy(logits.reshape(-1, 27), target_ids.reshape(-1), ignore_index=PAD_ID)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_loss = epoch_loss / len(train_loader)
    train_losses.append(train_loss)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs_b, ids_b, _ in val_loader:
            imgs_b, input_ids, target_ids = prepare_batch(imgs_b, ids_b)
            logits = model(imgs_b, input_ids)
            loss = F.cross_entropy(logits.reshape(-1, 27), target_ids.reshape(-1), ignore_index=PAD_ID)
            val_loss += loss.item()
    val_loss /= len(val_loader)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{N_EPOCHS} - train_loss {train_loss:.4f} - val_loss {val_loss:.4f} - time {time.time()-t0:.0f}s")

# Sauvegarde du checkpoint (important pour Colab qui peut couper)
torch.save(model.state_dict(), "model_baseline.pt")
print("Modèle sauvegardé.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.plot(train_losses, label="train loss")
plt.plot(val_losses, label="val loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Training / Validation Loss")
plt.legend()
plt.savefig("loss_curve.png")
plt.show()